# frp — Fast Reverse Proxy

frp exposes local services behind NAT using a server (frps) on a public host and a client (frpc) behind the firewall.

## Generate frps.toml

In [ ]:
def frps_config(bind_port=7000, dashboard_port=7500, vhost_http=8080,
                auth_token="change-this-token", log_level="info"):
    return f"""# frp server configuration
bindPort = {bind_port}

webServer.addr = "0.0.0.0"
webServer.port = {dashboard_port}
webServer.user = "admin"
webServer.password = "changeme"

auth.method = "token"
auth.token  = "{auth_token}"

vhostHTTPPort = {vhost_http}

log.to    = "console"
log.level = "{log_level}"
"""

print(frps_config())

## Generate frpc.toml

In [ ]:
def frpc_config(server_addr, server_port=7000, auth_token="change-this-token", proxies=None):
    lines = [
        f'serverAddr = "{server_addr}"',
        f"serverPort = {server_port}",
        "",
        'auth.method = "token"',
        f'auth.token  = "{auth_token}"',
        "",
        'log.to    = "console"',
        'log.level = "info"',
    ]
    for proxy in (proxies or []):
        lines += [
            "",
            "[[proxies]]",
            f'name      = "{proxy["name"]}"',
            f'type      = "{proxy["type"]}"',
            f'localIP   = "{proxy.get("localIP","127.0.0.1")}"',
            f'localPort = {proxy["localPort"]}',
            f'remotePort = {proxy["remotePort"]}',
        ]
    return "\n".join(lines)

cfg = frpc_config(
    server_addr="frps.example.com",
    proxies=[
        {"name": "web",   "type": "tcp", "localPort": 3000, "remotePort": 13000},
        {"name": "ssh",   "type": "tcp", "localPort": 22,   "remotePort": 16022},
        {"name": "mysql", "type": "tcp", "localPort": 3306, "remotePort": 13306},
    ]
)
print(cfg)

## Proxy type reference

In [ ]:
proxy_types = [
    ("tcp",   "Any TCP service — raw port forwarding"),
    ("udp",   "UDP services — DNS, game servers"),
    ("http",  "HTTP with virtual-host routing (needs vhostHTTPPort on frps)"),
    ("https", "HTTPS passthrough"),
    ("stcp",  "Secure TCP — peer-to-peer, no public port exposed"),
    ("xtcp",  "P2P UDP hole punching (for low-latency)"),
]
print(f"{'Type':<8}  Description")
print("-" * 60)
for t, desc in proxy_types:
    print(f"{t:<8}  {desc}")